# Blitz 遊戲機器人 (電腦適配版)

本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。
使用前請先執行下方的校正工具，找出你螢幕上遊戲棋盤的位置 (basex, basey, gap).

In [1]:
import pyautogui as pyg
import time

# 安全設定：當滑鼠移到螢幕角落時強制中止程式
pyg.FAILSAFE = True
pyg.PAUSE = 0.01

# ==========================================
# 已校正完成的電腦參數設定
# ==========================================
basex = 491
basey = 439
gap = 60

### 快速測試：讓滑鼠自動移動到第一顆寶石中心
執行下方區塊，滑鼠將會自動移動到 `(basex, basey)` 的位置，方便你確認座標是否正確對準。

In [2]:
print("準備移動滑鼠到第一顆寶石中心點...")
time.sleep(2)
pyg.moveTo(basex, basey, duration=0.5)
print(f"滑鼠已移動至座標: ({basex}, {basey})")

準備移動滑鼠到第一顆寶石中心點...
滑鼠已移動至座標: (491, 439)


### 步驟 1：滑鼠即時位置與顏色檢測器
執行下方儲存格後，將滑鼠移動到遊戲畫面第一顆寶石的正中央，記下座標。

In [3]:
print("請在 5 秒內將滑鼠移到遊戲棋盤左上角第一顆寶石中心...")
time.sleep(5)
x, y = pyg.position()
im = pyg.screenshot()
color = im.getpixel((x, y))
print(f"當前滑鼠位置: x={x}, y={y}, 顏色={color}")

請在 5 秒內將滑鼠移到遊戲棋盤左上角第一顆寶石中心...
當前滑鼠位置: x=491, y=439, 顏色=(16, 138, 253)


In [4]:
def gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心點的螢幕座標 (x, y)。"""
    return basex + c * gap, basey + r * gap

In [5]:
def get_gem_color(r, c, im):
    """回傳截圖中第 r 列、第 c 欄寶石中心點的顏色。"""
    x, y = gem_position(r, c)
    return im.getpixel((x, y))

In [6]:
def _can_move_in_direction(r, c, im, dr, dc):
    """判斷交換指定方向相鄰寶石後，是否會形成三連線。"""
    next_r, next_c = r + dr, c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= next_r < 8 and 0 <= next_c < 8):
        return False

    current_color = get_gem_color(r, c, im)
    next_color = get_gem_color(next_r, next_c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return next_color
        if (row, col) == (next_r, next_c):
            return current_color
        return get_gem_color(row, col, im)

    for center_r, center_c in ((r, c), (next_r, next_c)):
        for match_dr, match_dc in ((0, 1), (1, 0)):
            for start in (-2, -1, 0):
                positions = [
                    (center_r + (start + i) * match_dr, center_c + (start + i) * match_dc)
                    for i in range(3)
                ]
                if all(0 <= row < 8 and 0 <= col < 8 for row, col in positions):
                    colors = [color_after_swap(row, col) for row, col in positions]
                    if colors[0] == colors[1] == colors[2]:
                        return True

    return False

def can_move_left(r, c, im):
    return _can_move_in_direction(r, c, im, 0, -1)

def can_move_right(r, c, im):
    return _can_move_in_direction(r, c, im, 0, 1)

def can_move_up(r, c, im):
    return _can_move_in_direction(r, c, im, -1, 0)

def can_move_down(r, c, im):
    return _can_move_in_direction(r, c, im, 1, 0)

In [7]:
print("機器人即將開始，請切換至遊戲畫面...")
time.sleep(3)

print(f"起點座標檢查 -> basex: {basex}, basey: {basey}")

start_time = time.monotonic()
im = pyg.screenshot()
directions = (
    (can_move_left, 0, -1),
    (can_move_right, 0, 1),
    (can_move_up, -1, 0),
    (can_move_down, 1, 0),
)

while True:
    if time.monotonic() - start_time >= 60:
        print("時間到，機器人停止。")
        break

    moved = False
    for r in range(8):
        if time.monotonic() - start_time >= 60:
            break
        for c in range(8):
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = gem_position(r, c)
                    x2, y2 = gem_position(r + dr, c + dc)
                    
                    # ⚡ 乾淨快速的點擊與拖曳動作
                    pyg.moveTo(x1, y1)
                    pyg.mouseDown(button="left")
                    pyg.moveTo(x2, y2)
                    pyg.mouseUp(button="left")
                    
                    time.sleep(0.02)
                    im = pyg.screenshot()
                    moved = True
                    break
            if moved: break
        if moved: break

    if not moved:
        time.sleep(0.05)
        im = pyg.screenshot()

機器人即將開始，請切換至遊戲畫面...
起點座標檢查 -> basex: 491, basey: 439
時間到，機器人停止。
